In [0]:
reco_max = f"""
SELECT id_recommendation, batch, status, target_localization, calculation_date
FROM {source}.recommendations
WHERE deleted = false
"""

df_reco_max = spark.sql(reco_max)

# Grouper par 'batch_number' et 'target_localization', pour compter les occurrences
df_counts = df_reco_max.groupBy("batch", "target_localization").count()

# Joindre le DataFrame original avec les comptes pour ajouter une nouvelle colonne
df_reco_max = df_reco_max.join(df_counts, on=["batch", "target_localization"], how="left")

# Renommer la colonne 'count'
df_reco_max = df_reco_max.withColumnRenamed("count", "batch_target_count")

# Modifier la colonne 'status' en fonction de la valeur de 'batch_target_count'
df_reco_max = df_reco_max.withColumn(
    "status",
    F.when(F.col("batch_target_count") == 1, F.lit(1)).otherwise(F.col("status"))
)

# Filtrer les lignes où 'status' est égal à 1
df_reco_max = df_reco_max.filter(F.col("status") == 1)

# Grouper par 'batch_number' et 'target_localization', et prendre la date max de 'calculation_date'
df_reco_max = df_reco_max.groupBy("batch", "target_localization") \
                        .agg(F.max("calculation_date").alias("max_calculation_date"))

# Ajouter une colonne "Reco - Date Max" avec la valeur 1
df_reco_max = df_reco_max.withColumn("Reco - Date Max", F.lit(1))

In [0]:
# Joindre df_reco avec df_reco_max en utilisant les colonnes spécifiées
df_reco = df_reco.join(df_reco_max,
                        on=[df_reco.batch == df_reco_max.batch,
                            df_reco.target_localization == df_reco_max.target_localization,
                            df_reco.calculation_date == df_reco_max.max_calculation_date],
                        how="left")

df_reco = df_reco.drop(df_reco_max.batch) \
                   .drop(df_reco_max.target_localization) \
                   .drop(df_reco_max.max_calculation_date)

# Filtrer les lignes où "Reco - Date Max" est égal à 1
df_reco = df_reco.filter(F.col("Reco - Date Max") == 1)

# Ajouter la colonne 'status' en fonction de la valeur de 'acceptance_status'
df_reco = df_reco.withColumn(
    "acceptance_status",
    F.when(F.col("applied_status") == "refusee_ia", -1)
     .when(F.col("applied_status") == "refusee_usine", -2)
     .when(F.col("applied_status") == "en_attente", 0)
     .when(F.col("applied_status") == "acceptee_mais_non_appliquee", 1)
     .when(F.col("applied_status") == "acceptee_et_appliquee", 2))

# Ajout de la colonne avec la date du jour
df_reco = df_reco.withColumn("today", current_date())

# Calcul du nombre de jours entre date_du_jour et calculation_date
df_reco = df_reco.withColumn("days_before", datediff(col("today"), col("calculation_date")))

# Créer une vue temporaire pour le DataFrame
df_reco.createOrReplaceTempView("reco")

In [0]:
# Ajout des notes de production
reco_notes = f"""
SELECT bn.batch, bn.note AS note_1, pa.login AS user_note_1
FROM {source}.batches_notes bn
JOIN {source}.profiles_accounts pa ON pa.id_profile_account = bn.profile_account AND pa.deleted = false
WHERE bn.deleted = false
"""

df_reco_notes = spark.sql(reco_notes)

In [0]:
# Ajout d'un numéro de ligne pour différencier les doublons
window_spec = Window.partitionBy("batch").orderBy("batch")
df_with_rownum = df_reco_notes.withColumn("row_num", row_number().over(window_spec))

# Séparer les doublons en note_2 et user_note_2
df_notes_2 = df_with_rownum.filter(col("row_num") == 2).select(
    col("batch").alias("batch_2"),
    col("note_1").alias("note_2"),
    col("user_note_1").alias("user_note_2")
)

# Séparer les doublons en note_3 et user_note_3
df_notes_3 = df_with_rownum.filter(col("row_num") == 3).select(
    col("batch").alias("batch_3"),
    col("note_1").alias("note_3"),
    col("user_note_1").alias("user_note_3")
)

# Garder uniquement la première occurrence
df_notes_1 = df_with_rownum.filter(col("row_num") == 1).drop("row_num")

# Fusionner les deux dataframes
df_reco_notes = df_notes_1.join(df_notes_2, df_notes_1.batch == df_notes_2.batch_2, "left") \
    .drop("batch_2")
df_reco_notes = df_reco_notes.join(df_notes_3, df_reco_notes.batch == df_notes_3.batch_3, "left") \
    .drop("batch_3")

# Créer une vue temporaire pour le DataFrame
df_reco_notes.createOrReplaceTempView("reco_notes")

In [0]:
reco = f"""
SELECT reco.*, reco_notes.note_1, reco_notes.user_note_1, reco_notes.note_2, reco_notes.user_note_2, reco_notes.note_3, reco_notes.user_note_3
FROM reco
LEFT JOIN reco_notes ON reco_notes.batch = reco.batch
"""

df_reco = spark.sql(reco)

In [0]:
# Supprimer les colonnes suivantes
df_reco = df_reco.select("id_recommendation", "batch", "calculation_date", "acceptance_user", "acceptance_comments", "target_localization", "acceptance_status", "days_before", "note_1", "user_note_1", "note_2", "user_note_2", "note_3", "user_note_3")

# Remplir les cellules vides des colonnes avec "-"
df_reco = df_reco.fillna("-", subset=["note_1", "user_note_1", "note_2", "user_note_2", "note_3", "user_note_3"])

In [0]:
current_process="fact_recommendations"

In [0]:
target_fact_recommendations = current_catalog +"."+current_schema+"."+current_process
print(target_fact_recommendations)

In [0]:
all_columns =  df_reco.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_recommendation']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_reco, 
    target_fact_recommendations, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )